# Task 1 (Entrega Parcial): Diseno formal del MDP

**Dominio:** una empresa de logistica urbana opera una flota de drones autonomos que recogen paquetes en puntos de origen y los entregan en direcciones dentro de una ciudad. La ciudad se abstrae como una grilla 3D de celdas (bloques urbanos x, y, y niveles de altitud z) que incluye edificios, zonas de vuelo restringido, estaciones de carga y puntos de entrega. Cada mision de entrega se modela como un MDP $(\mathcal{S}, \mathcal{A}, p, r, \gamma)$ para un unico dron.

## 1. Espacio de estados $\mathcal{S}$

Proponemos un estado compuesto (vector de variables discretizadas):

$$s = (x, y, z,\ b,\ \text{pkg},\ \text{dest},\ w,\ \text{rain},\ c,\ \tau,\ \text{op})$$

| Variable | Descripcion | Justificacion de inclusion |
|---|---|---|
| $(x,y,z)$ | Posicion del dron en la grilla urbana | Es la variable minima necesaria para decidir la siguiente accion de movimiento; sin ella el modelo no podria predecir a donde lleva cada accion. |
| $b$ | Nivel de bateria (bins de 5%) | El costo y riesgo de cada accion futura (p. ej. si conviene cargar o si un vuelo es viable) depende del presupuesto de energia restante, no de cuanta energia se gasto en el pasado. |
| $\text{pkg}$ | Estado del paquete: {sin_recoger, en_transporte, entregado} | Determina que acciones (`Recoger`, `Entregar`) son validas y define el evento que genera la recompensa principal. |
| $\text{dest}$ | Identificador del destino asignado | Sin esta variable el dron no sabria hacia que celda dirigirse; es el objetivo de la mision actual. |
| $w$ | Viento local (calma/moderado/fuerte) en la celda actual | Afecta directamente la probabilidad de deriva en la transicion y el consumo de bateria del siguiente paso. |
| $\text{rain}$ | Precipitacion local (discretizada) | Afecta la probabilidad de fallo/riesgo de seguridad del siguiente movimiento. |
| $c$ | Nivel de congestion aerea en la celda/vecindad | Aproxima el efecto de otros drones de la flota sin modelar explicitamente sus posiciones (ver Task 2, pregunta 1). |
| $\tau$ | Tiempo restante hasta el limite de entrega | Necesario para que la recompensa pueda distinguir una entrega a tiempo de una tardia, y para que decisiones como "cargar ahora vs. seguir" sean evaluables. |
| $\text{op}$ | Estado operativo del dron {en_vuelo, aterrizado, cargando, en_falla} | Determina que acciones estan disponibles y si el episodio debe terminar (falla). |

**Variables que se omiten deliberadamente y su consecuencia:**

- **Historial completo de la trayectoria pasada** (ruta seguida hasta llegar a $s$): se omite porque, si el desgaste acumulado relevante ya esta resumido en $b$ y en $\text{op}$, la ruta pasada no aporta informacion adicional para predecir el futuro. *Consecuencia si esto fuera falso:* si hay desgaste mecanico dependiente de la ruta (p. ej. numero de giros bruscos) que no se refleja en $\text{op}$, el modelo perderia precision y violaria Markov de forma sutil (ver Task 2).
- **Posicion exacta de los demas drones de la flota**: se omite y se reemplaza por la variable agregada $c$ (congestion). *Consecuencia:* se pierde la capacidad de anticipar maniobras especificas de otros drones (p. ej. una colision inminente con un dron concreto); el modelo solo captura el riesgo promedio de la zona, no el riesgo instantaneo exacto.
- **Mapa meteorologico de toda la ciudad** (solo se usa la lectura local $w$, $\text{rain}$): se omite por tamano de estado. *Consecuencia:* el agente es miope ante frentes de tormenta que se aproximan pero aun no han llegado a su celda; podria entrar en una zona que se volvera peligrosa un paso despues sin haberlo anticipado.
- **Identidad del operador humano / metadatos administrativos del pedido** (cliente, precio, etc.): se omiten por ser irrelevantes para la dinamica fisica y de decision de vuelo -- no afectan $p(s'\mid s,a)$.

## 2. Espacio de acciones $\mathcal{A}$

Se define un espacio de acciones **discreto**:

$$\mathcal{A} = \{\text{Norte, Sur, Este, Oeste, Subir, Bajar, Hover, Recoger, Entregar, Cargar, AterrizajeEmergencia}\}$$

**Justificacion de discretizacion:** a nivel de decision logistica (a que celda moverse, cuando recoger/entregar/cargar) un espacio discreto sobre una grilla de waypoints es suficiente y mucho mas tratable que control continuo de velocidades/angulos; el control de bajo nivel (estabilizacion, trayectoria fisica entre dos celdas) se delega a un controlador separado (p. ej. PID) fuera del alcance del agente de RL -- una descomposicion jerarquica habitual en robotica aerea.

**Restricciones de acciones por estado** -- se modelan mediante una funcion de acciones validas $\mathcal{A}(s) \subseteq \mathcal{A}$ (action masking):

- Movimientos hacia celdas de **zona de vuelo restringida** (aeropuertos, espacio aereo prohibido) quedan excluidos de $\mathcal{A}(s)$: es una restriccion fisica/legal dura, se modela por *enmascaramiento* (la accion ni siquiera esta disponible), no por penalizacion, para garantizar que nunca se elija.
- `Recoger` solo pertenece a $\mathcal{A}(s)$ si el dron esta en la celda de origen y $\text{pkg}=\text{sin\_recoger}$; `Entregar` solo si esta en la celda destino y $\text{pkg}=\text{en\_transporte}$.
- `Cargar` solo disponible si la celda actual es una estacion de carga.
- Si $b$ cae por debajo de un umbral critico (p. ej. 10%), $\mathcal{A}(s)$ se restringe a $\{\text{Cargar (si aplica)}, \text{AterrizajeEmergencia}, \text{Bajar hacia estacion mas cercana}\}$: en vez de prohibir el vuelo normal por completo, se deja que el riesgo se exprese de forma *blanda* a traves de $p(s'\mid s,a)$ (mayor probabilidad de fallo/caida), ya que agotar la bateria es un proceso fisico gradual y estocastico, no un limite abrupto.
- `AterrizajeEmergencia` esta siempre disponible en $\mathcal{A}(s)$ como accion de seguridad universal.

## 3. Funcion de recompensa $r(s,a,s')$

Se disena como una combinacion ponderada de tres componentes que capturan los objetivos (potencialmente conflictivos) de eficiencia, bateria y seguridad, mas un costo de paso:

$$r(s,a,s') = r_{\text{entrega}}(s,a,s') + r_{\text{bateria}}(s,a,s') + r_{\text{seguridad}}(s,a,s') + r_{\text{tiempo}}$$

- $r_{\text{entrega}}$: $+100$ si $s'$ marca $\text{pkg}=\text{entregado}$ dentro del plazo ($\tau \geq 0$); $+40$ si se entrega mas tarde del plazo (se sigue prefiriendo entregar tarde a no entregar); $0$ en otro caso.
- $r_{\text{bateria}}$: $-0.1 \times \Delta b$ por cada unidad de bateria consumida en el paso (penaliza maniobras derrochadoras como subir/bajar innecesariamente u *hover* prolongado); $-200$ si $s'$ tiene $\text{op}=\text{en\_falla}$ por agotamiento de bateria (caida).
- $r_{\text{seguridad}}$: $-150$ si $s'$ corresponde a una violacion de espacio restringido o colision; $-20 \times \text{severidad}(w,\text{rain})$ si se vuela bajo condiciones climaticas severas (proporcional a la severidad, ya que el riesgo real de accidente crece con el clima).
- $r_{\text{tiempo}} = -1$ en cada paso (costo constante) para incentivar rutas cortas / eficientes en tiempo.

**Ponderacion:** se elige deliberadamente la jerarquia de magnitudes

$$|r_{\text{crash}}| > |r_{\text{noflyzone}}| > r_{\text{entrega\_a\_tiempo}} > r_{\text{entrega\_tardia}} \gg \text{costo de bateria por paso} > |r_{\text{tiempo}}|$$

de modo que **ninguna combinacion de ahorros de tiempo o bateria a lo largo de un episodio pueda compensar una unica violacion de seguridad**: la seguridad debe dominar estrictamente sobre la eficiencia y el consumo energetico.

**Consecuencias de una ponderacion incorrecta:**

- Si el costo de bateria ($-0.1\Delta b$) es demasiado alto en relacion a $r_{\text{entrega}}$, el agente puede aprender una politica degenerada que evita moverse (o aborta misiones) para minimizar consumo, sacrificando la tasa de entregas -- un caso de *reward hacking* hacia el objetivo equivocado.
- Si $r_{\text{tiempo}}$ (costo por paso) es demasiado negativo respecto a $r_{\text{seguridad}}$, el agente puede aprender a atravesar zonas restringidas o volar en clima severo para ahorrar pasos, priorizando velocidad sobre seguridad.
- Si la diferencia entre entrega a tiempo y tardia es demasiado pequena, el agente ignora los plazos de entrega, un objetivo de negocio central para la empresa.
- Si toda la senal de recompensa depende solo del evento terminal (entrega) sin costo de paso ni shaping, el aprendizaje es muy lento por *sparsity* de recompensa, especialmente en grillas urbanas grandes.

## 4. Funcion de transicion $p(s' \mid s,a)$

Es **estocastica**. Fuentes reales de aleatoriedad en el dominio: viento que desvia al dron de la celda objetivo, fallos mecanicos/de sensores, variabilidad en el consumo de bateria, fallas de acoplamiento en estaciones de carga, y errores de manipulacion en la entrega fisica del paquete.

**Transicion 1 -- `Este` en clima calmo, bateria > 20%:**

$$p((x+1,y,z,\dots) \mid s,\text{Este}) = 0.90,\quad p((x+1,y{+}1,z,\dots)\mid s,\text{Este}) = 0.05,\quad p((x+1,y{-}1,z,\dots)\mid s,\text{Este}) = 0.05$$

*Justificacion:* controladores de vuelo guiados por GPS logran alta precision en condiciones calmas, pero persiste un residual de ruido/viento leve que desplaza lateralmente al dron con baja probabilidad.

**Transicion 2 -- `Este` con $w=\text{fuerte}$:**

$$p(\text{intencionado}) = 0.60,\quad p(\text{deriva lateral } +y) = 0.20,\quad p(\text{deriva lateral } -y) = 0.15,\quad p(\text{fallo, permanece en } s) = 0.05$$

*Justificacion:* bajo viento fuerte aumenta considerablemente la probabilidad de desviacion lateral (empuje aerodinamico) y aparece una probabilidad no despreciable de fallo por sobrecarga del sistema de estabilizacion, ausente en clima calmo.

**Transicion 3 -- `Cargar` en estacion de carga:**

$$p(b' = \min(100, b+\Delta_{\text{carga}}) \mid s,\text{Cargar}) = 0.95,\qquad p(b'=b \mid s,\text{Cargar}) = 0.05$$

*Justificacion:* el proceso de carga es mecanicamente mas confiable que el vuelo, pero aun sujeto a fallos de acoplamiento o a que la estacion este ocupada por otro dron de la flota (congestion de infraestructura).

**Transicion 4 -- `Entregar` en destino correcto con paquete a bordo:**

$$p(\text{pkg}'=\text{entregado}) = 0.98, \qquad p(\text{pkg}'=\text{en\_transporte, sin cambio}) = 0.02$$

*Justificacion:* la entrega fisica tiene una probabilidad residual de fallo (destinatario ausente, error del mecanismo de liberacion del paquete).

## 5. Factor de descuento $\gamma$

Se propone $\gamma = 0.98$.

**Justificacion en terminos del problema (no solo matematicos):**

- La tarea es episodica (ver Task 2, pregunta 3) con episodios de longitud moderada-larga (cientos de pasos al navegar una grilla urbana). Un $\gamma$ demasiado bajo descontaria casi a cero la recompensa terminal de entrega ($r_{\text{entrega}}$), haciendo que el agente se vuelva miope y priorice ahorros inmediatos de bateria sobre completar la mision -- justo el comportamiento que se busca evitar.
- $\gamma=0.98$ conserva peso significativo sobre el horizonte tipico de un episodio: $0.98^{300}\approx 0.0024$, es decir, la recompensa de entrega sigue siendo relevante incluso a 300 pasos de distancia.
- No se elige $\gamma=1$ aunque el episodio este garantizado a terminar en tiempo finito (por agotamiento de bateria o limite de tiempo): un descuento levemente menor a 1 expresa una preferencia legitima del negocio por **completar la entrega cuanto antes** entre politicas igualmente exitosas (rutas mas cortas son preferibles), y ademas captura implicitamente que **mas tiempo en vuelo implica mas exposicion acumulada a riesgo** (clima, colisiones), reforzando el objetivo de seguridad.
- Si se adoptara la variante de flota operando de forma continua (ver Task 2, pregunta 3), seria necesario $\gamma$ estrictamente menor a 1 (p. ej. 0.95) para garantizar convergencia del retorno en horizonte infinito, o migrar a una formulacion de recompensa promedio (*average reward*).

---
# Task 2 (Entrega Parcial)

Respondan las siguientes preguntas con argumentacion tecnica, en base al diseno del MDP de la Task 1.

## 1. Violaciones de la propiedad de Markov

**Situacion 1 -- Clima con memoria temporal (rachas de viento / frentes de tormenta).**
El estado solo incluye la lectura instantanea de viento y lluvia en la celda actual ($w$, $\text{rain}$). Si el clima real tiene autocorrelacion temporal (una racha de viento no es independiente del paso anterior, o hay un frente de tormenta acercandose de forma predecible), entonces $P(w_{t+1}\mid w_t, w_{t-1},\dots) \neq P(w_{t+1}\mid w_t)$, y el estado actual deja de ser un resumen suficiente del pasado: se viola Markov.

*Extension propuesta:* agregar al estado una ventana de las ultimas $k$ lecturas de viento, o -- mas eficiente -- el estado interno de baja dimension de un filtro (p. ej. un filtro de Kalman o un modelo autorregresivo) que resuma tendencia/aceleracion del viento, en vez de guardar el historial crudo.

*Costo computacional:* guardar $k$ lecturas discretizadas en $L$ niveles multiplica el espacio de estados por $L^k$ (crecimiento exponencial en $k$); usar el estado resumido de un filtro mantiene el tamano del estado practicamente constante (una o dos variables continuas adicionales), a cambio de mayor costo de computo por paso (actualizar el filtro) y de perder la interpretacion tabular directa, lo que tipicamente obliga a usar aproximacion de funcion en vez de una tabla $Q(s,a)$.

**Situacion 2 -- Congestion aerea como variable agregada de otros agentes.**
El estado resume la posicion e intenciones de los demas drones de la flota en una sola variable $c$ (nivel de congestion). La probabilidad real de colision/retraso en el siguiente paso depende de las posiciones y rutas planeadas de esos otros drones, informacion que no esta en $s$. Si la congestion evoluciona de forma predecible (patrones de hora pico) mas alla de lo que la lectura actual de $c$ captura, el estado deja de ser suficiente: se viola Markov.

*Extension propuesta:* incluir en el estado las posiciones (o rutas planeadas) de los drones vecinos dentro de un radio $r$, o un mapa de congestion discretizado sobre la vecindad inmediata en vez de un unico escalar agregado.

*Costo computacional:* incluir posiciones exactas de $N$ drones vecinos multiplica el espacio de estados por (numero de celdas)$^N$ -- crecimiento exponencial en $N$, impracticable para flotas grandes. La alternativa de un mapa de congestion local (vecindad de tamano fijo) crece solo linealmente en el numero de celdas vecinas consideradas, sacrificando precision (no distingue que dron especifico genera el riesgo) a cambio de escalabilidad.

**Situacion 3 (adicional) -- Desgaste acumulado real de la bateria.**
El porcentaje de bateria reportado por el sensor no siempre refleja la capacidad real restante: dos drones con el mismo $b\%$ pero distinto historial de ciclos de carga/temperatura pueden comportarse de forma distinta ante la misma accion. Si $b$ no captura ese estado fisico (*state of health*), se viola Markov.

*Extension propuesta:* agregar una variable de salud de bateria (SoH) o numero de ciclos de carga, actualizada lentamente entre episodios. *Costo computacional:* bajo -- una unica variable discreta adicional con pocos niveles.

## 2. Observabilidad completa en logistica urbana

Asumir observabilidad completa **no es del todo razonable**; es una simplificacion conveniente adoptada al modelar el problema como MDP en vez de POMDP.

**Variables del estado real probablemente no observables directamente por el dron:**

- Posicion y trayectoria exacta de otros drones/obstaculos moviles fuera del rango o linea de vista de los sensores (de ahi que se use la variable agregada de congestion $c$ y no posiciones exactas).
- Condiciones meteorologicas fuera de la celda actual (viento/lluvia en la ruta futura todavia no medidos).
- El estado interno real de la bateria (quimica, *state of health*) -- solo se observa una estimacion indirecta ($b\%$) calculada por el sistema de gestion de bateria, con error.
- Posicion exacta del propio dron: en entornos urbanos con edificios altos, el GPS sufre el efecto de *urban canyon* (multipath, perdida de senal), por lo que $(x,y,z)$ observado es en realidad una estimacion ruidosa de la posicion real.
- Disponibilidad real de la estacion de carga o del punto de entrega en el instante exacto de llegada (p. ej. otro dron ocupandola) si no hay comunicacion perfecta en tiempo real.
- Fallas incipientes de hardware (degradacion de motor) no detectadas aun por los sensores.

**Efecto sobre la validez MDP vs. POMDP:**

Un MDP asume que la observacion disponible para el agente **es** el estado real y suficiente estadisticamente. Si en realidad el dron solo tiene acceso a una observacion ruidosa/parcial $o_t = g(s_t) + \epsilon$ (lo cual es el caso aqui), el problema es formalmente un **POMDP**: la decision optima deberia basarse en una distribucion de creencia (*belief state*) sobre los estados posibles dado el historial de observaciones, no en la observacion cruda de un solo instante.

Tratarlo como MDP (usar la lectura de sensores directamente como si fuera el estado) es una aproximacion razonable cuando el ruido es pequeno y las variables no observadas tienen bajo impacto en la dinamica relevante -- justificable, por ejemplo, para posicion y bateria con buena instrumentacion. Se degrada, en cambio, cuando la incertidumbre es alta (clima cambiante, comportamiento de otros agentes), caso en el que conviene mantener explicitamente un *belief state* (POMDP) o al menos una aproximacion intermedia (*belief MDP* con un filtro/historial corto embebido en el estado), aceptando el costo computacional adicional que eso implica.

## 3. ¿Tarea episodica o continua?

**Argumento:** este problema deberia modelarse como **episodico**, con un episodio = una mision de entrega de un paquete. Existen estados terminales naturales y bien definidos: entrega exitosa ($\text{pkg}=\text{entregado}$), falla/caida ($\text{op}=\text{en\_falla}$), o agotamiento del plazo/bateria sin completar la mision. Operacionalmente, la empresa tambien gestiona el negocio en unidades discretas de "ruta" o "pedido", no como un proceso ininterrumpido sin reinicio.

Una vista alternativa **continua** seria razonable solo si se modela la operacion agregada de toda la flota a lo largo del dia como un unico proceso perpetuo (el dron entrega, vuelve a base, recarga y encadena la siguiente entrega sin un punto de reinicio natural) -- mismo agente, sin estados absorbentes.

**Como cambia el diseno segun la decision:**

| | Episodica (recomendado) | Continua |
|---|---|---|
| **Recompensa** | Puede anclarse a un evento terminal claro ($r_{\text{entrega}}$ grande al llegar a un estado absorbente); el retorno por episodio es finito por construccion. | No hay un evento terminal unico que reiniciar; se necesita una formulacion de tasa (p. ej. recompensa promedio por unidad de tiempo, o $+R$ repetible cada vez que se completa una entrega dentro de un flujo continuo), evitando que la suma de recompensas diverja. |
| **$\gamma$** | Puede ser alto, incluso cercano a 1 (aqui se propuso 0.98), porque el horizonte es finito y la suma de recompensas esta garantizada acotada. | Debe ser estrictamente menor a 1 (p. ej. 0.90-0.95) para que el retorno descontado converja en horizonte infinito, o bien sustituirse por una formulacion de *average reward* en vez de recompensa descontada. |

**Conclusion:** se recomienda la formulacion episodica porque es mas natural para el dominio, produce objetivos de exito/fracaso claros por entrega, facilita el entrenamiento (retornos de Monte Carlo por episodio) y da metricas de negocio directamente interpretables (tasa de exito, tiempo promedio y consumo por entrega). La vista continua solo aportaria valor si el objetivo fuera optimizar el comportamiento agregado de la flota en operacion 24/7 sin puntos de reinicio identificables entre entregas.